<a href="https://colab.research.google.com/github/Doris-QZ/Fine-Tuning-BERT-Large-with-QLoRA-for-Author-Identification/blob/main/3_Ensemble_Results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Introduction

This notebook is the final part of the project on fine-tuning the **BERT Large** model using **QLoRA** for the  [**Spooky Author Identification**](https://www.kaggle.com/competitions/spooky-author-identification) dataset from Kaggle.  

It ensembles the two best models--[`QLoRA_r8a4_AllLin_Author_Identification.ipynb`](https://github.com/Doris-QZ/Fine-Tuning-BERT-Large-with-QLoRA-for-Author-Identification/blob/main/1_QLoRA_r8a4_AllLin_Author_Identification.ipynb) and [`QLoRA_r16a8_QKV_Author_Identification.ipynb`](https://github.com/Doris-QZ/Fine-Tuning-BERT-Large-with-QLoRA-for-Author-Identification/blob/main/2_QLoRA_r16a8_QKV_Author_Identification.ipynb)--to generate predictions on the validation and test sets.

<br>

**Data Source**: Meg Risdal and Rachael Tatman. Spooky Author Identification. https://kaggle.com/competitions/spooky-author-identification, 2017. Kaggle.


**Connect to Kaggle and download the dataset**

In [ ]:
# Install Kaggle library
!pip install kaggle

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Make a directory for Kaggle
!mkdir ~/.kaggle

# Copy the kaggle.json file to the directory
!cp /content/drive/MyDrive/ColabNotebooks/Kaggle_API_Key/kaggle.json ~/.kaggle/

# Change the file permission to read/write to the owner only
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset
!kaggle competitions download spooky-author-identification

# Unzip the data
!unzip spooky-author-identification.zip
!unzip train.zip
!unzip test.zip

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate peft

In [3]:
import bitsandbytes
import transformers
import peft
import accelerate

print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"accelerate: {accelerate.__version__}")

bitsandbytes: 0.48.1
transformers: 4.57.0
peft: 0.17.1
accelerate: 1.10.1


In [4]:
# Import important packages
import pandas as pd
import numpy as np

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, BertForSequenceClassification, BitsAndBytesConfig
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from peft import PeftModel

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

### Dataset Preparation

In [5]:
# Load the data
train = pd.read_csv('/content/train.csv')
test = pd.read_csv('/content/test.csv')

# Create a new column of 'label' to encode 'author' to numeric
label_to_id = {'EAP': 0, 'MWS': 1, 'HPL': 2}
train['labels'] = train['author'].map(label_to_id)

# Split the 'train' data into training and validation dataset
train_set, val_set = train_test_split(train, test_size=0.1, random_state=42)

In [ ]:
# Load the pretrained tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-large-uncased")

# Tokenize text data
val_tokenized = tokenizer(val_set['text'].tolist(),
                          max_length=512,
                          truncation=True,
                          padding=False)

test_tokenized = tokenizer(test['text'].tolist(),
                           max_length=512,
                          truncation=True,
                          padding=False)

In [7]:
class TextDataset(torch.utils.data.Dataset):
    """
    Create a PyTorch Dataset object from tokenized text data and labels.

    Args:
        data (dict): Dictionary of lists from deberta-v3-large tokenizer(without padding)
        label (list): None or List of numerical labels

    Returns:
        dict[str, torch.tensor]: A dictionary containing a single sample's tensors.

    """
    def __init__(self, data, label=None):
        self.data = data
        self.label = label

    def __len__(self):
        return len(self.data['input_ids'])

    def __getitem__(self, idx):
        item = {'input_ids': torch.tensor(self.data['input_ids'][idx]),
                'token_type_ids': torch.tensor(self.data['token_type_ids'][idx]),
                'attention_mask': torch.tensor(self.data['attention_mask'][idx])}
        if self.label:
            item['labels'] = torch.tensor(self.label[idx])
        return item


# Create validation and test dataset
val_dataset = TextDataset(val_tokenized, val_set['labels'].tolist())
test_dataset = TextDataset(test_tokenized)

### Ensemble

In [ ]:
# Set quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load two quantized base models for two QLoRA models
base_model_1 = BertForSequenceClassification.from_pretrained(
    "bert-large-uncased",
    num_labels=3,
    quantization_config=quantization_config,
    device_map='auto'
)

base_model_2 = BertForSequenceClassification.from_pretrained(
    "bert-large-uncased",
    num_labels=3,
    quantization_config=quantization_config,
    device_map='auto'
)

In [9]:
# Load Adapter Weights to the two QLoRA models
r8a4_AllLin_checkpoint = '/content/drive/MyDrive/ColabNotebooks/Spooky_Author_Identification/qlora/r8a4_AllLin/checkpoint-1380'
r8a4_AllLin = PeftModel.from_pretrained(base_model_1, r8a4_AllLin_checkpoint)

r16a8_QKV_checkpoint = '/content/drive/MyDrive/ColabNotebooks/Spooky_Author_Identification/qlora/r16a8QKV_3/checkpoint-1656'
r16a8_QKV = PeftModel.from_pretrained(base_model_2, r16a8_QKV_checkpoint)

In [10]:
# Create data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

# Define dummy training arguments
AllLin_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/ColabNotebooks/Spooky_Author_Identification/qlora/ensemble_result',
    per_device_eval_batch_size=16,
    report_to = "none"
)

QKV_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/ColabNotebooks/Spooky_Author_Identification/qlora/ensemble_result',
    per_device_eval_batch_size=16,
    report_to = "none"
)

# Create the trainer
trainer_AllLin = Trainer(
    model=r8a4_AllLin,
    args=AllLin_args,
    data_collator=data_collator
)

trainer_QKV = Trainer(
    model=r16a8_QKV,
    args=QKV_args,
    data_collator=data_collator
)

r8a4_AllLin.config.use_cache = True
r16a8_QKV.config.use_cache = True

In [11]:
# Make predictions on the validation set
pred_AllLin = trainer_AllLin.predict(val_dataset)
pred_QKV = trainer_QKV.predict(val_dataset)

In [12]:
def ensemble_predict(pred_model1, pred_model2, weights=[0.5, 0.5]):
    # Convert Numpy logits to PyTorch tensors
    logits_model1 = torch.tensor(pred_model1.predictions, dtype=torch.float32)
    logits_model2 = torch.tensor(pred_model2.predictions, dtype=torch.float32)

    # Convert logits to probabilities
    probs_model1 = F.softmax(logits_model1, dim=1)
    probs_model2 = F.softmax(logits_model2, dim=1)

    # Calculate the weighted ensemble probability
    ensemble_probs = (weights[0] * probs_model1) + (weights[1] * probs_model2)

    # Get the final predicted class
    ensemble_preds = torch.argmax(ensemble_probs, dim=1).numpy()

    return ensemble_probs, ensemble_preds

In [13]:
# Get ensemble probability and predicted class of the validation set
val_probs, val_preds = ensemble_predict(pred_AllLin, pred_QKV, weights=[0.55, 0.45])

In [14]:
# Get the true labels
true_labels = torch.tensor(pred_AllLin.label_ids, dtype=torch.long)

# Calcuate validation accuracy
ensemble_accu = accuracy_score(true_labels, val_preds)
print(f"Ensemble Validation Accuracy: {ensemble_accu: .4f}")

# Compute the log probabilities
log_probs = torch.log(val_probs + 1e-12) # add epsilon to avoid log(0)

# Compute validation loss
loss = F.nll_loss(log_probs, true_labels)
print(f"Ensemble Validation Loss: {loss: .4f}")

Ensemble Validation Accuracy:  0.8636
Ensemble Validation Loss:  0.3216


In [15]:
# Make prediction on the test set
pred_test_AllLin = trainer_AllLin.predict(test_dataset)
pred_test_QKV = trainer_QKV.predict(test_dataset)

# Get ensemble probability and predicted class of the test set
test_probs, test_preds = ensemble_predict(pred_test_AllLin, pred_test_QKV, weights=[0.55, 0.45])

test_probs

tensor([[1.2581e-02, 9.5959e-01, 2.7826e-02],
        [9.7357e-01, 1.6606e-03, 2.4774e-02],
        [1.2869e-03, 4.5589e-04, 9.9826e-01],
        ...,
        [8.6175e-01, 3.9366e-03, 1.3432e-01],
        [5.0859e-01, 4.6218e-01, 2.9229e-02],
        [2.0508e-03, 2.2688e-04, 9.9772e-01]])

In [16]:
# Create the dataframe for Kaggle submission
ensemble_preds = pd.DataFrame(test_probs.numpy(), columns=['EAP', 'MWS', 'HPL'])
ensemble_preds = pd.concat([test['id'], ensemble_preds], axis=1)
ensemble_preds = ensemble_preds[['id', 'EAP', 'HPL', 'MWS']]
ensemble_preds.head()

,id,EAP,HPL,MWS
0,id02310,0.012581,0.027826,0.959594
1,id24541,0.973566,0.024774,0.001661
2,id00134,0.001287,0.998257,0.000456
3,id27757,0.620305,0.377942,0.001753
4,id04081,0.570896,0.406534,0.022570


In [17]:
ensemble_preds.to_csv('/content/drive/MyDrive/ColabNotebooks/Spooky_Author_Identification/qlora/ensemble_preds.csv', index=False)

After submitting to Kaggle, we obtained a private score of **0.33 (log loss)** on the test dataset--an improvement of **0.24** compared to the **BERT Base** model fine-tuned on the last few layers, which achieved a log loss of **0.57**.